# 09B – Batch Prediction Pipeline (Enterprise)

Generate bankruptcy predictions for multiple companies using the finalized production model.

## Business Objective
Build a production-style batch inference pipeline capable of scoring multiple companies at once and exporting the results.

In [1]:
import pandas as pd
import joblib

MODEL_PATH='../models/production_bankruptcy_model.joblib'
INPUT_DATA='../data/datasets/american_bankruptcy_cleaned.csv'

model=joblib.load(MODEL_PATH)
df=pd.read_csv(INPUT_DATA)


In [2]:
target='status_label' if 'status_label' in df.columns else 'target'

if target in df.columns:
    X=df.drop(columns=[target])
if 'company_name' in X.columns:
    X=X.drop(columns=['company_name'])
else:
    X=df.copy()

predictions=model.predict(X)
probabilities=model.predict_proba(X)[:,1]


In [3]:
results=X.copy()

results['Prediction']=predictions
results['Bankruptcy_Probability']=probabilities

results['Prediction_Label']=results['Prediction'].map({
    0:'Healthy',
    1:'Bankrupt'
})

results.to_csv('batch_prediction_results.csv',index=False)

results.head()


,year,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X12,X13,X14,X15,X16,X17,X18,Prediction,Bankruptcy_Probability,Prediction_Label
0,1999,511.267,833.107,18.373,89.031,336.018,35.163,128.348,372.7519,1024.333,...,70.658,191.226,163.816,201.026,1024.333,401.483,935.302,0,0.071765,Healthy
1,2000,485.856,713.811,18.577,64.367,320.590,18.531,115.187,377.1180,874.255,...,45.790,160.444,125.392,204.065,874.255,361.642,809.888,0,0.094354,Healthy
2,2001,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,364.5928,638.721,...,4.711,112.244,150.464,139.603,638.721,399.964,611.514,0,0.259345,Healthy
3,2002,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,143.3295,606.337,...,3.573,109.590,203.575,124.106,606.337,391.633,575.592,0,0.273606,Healthy
4,2003,432.204,523.302,26.680,47.491,247.245,3.504,104.661,308.9071,651.958,...,20.811,128.656,131.261,131.884,651.958,407.608,604.467,0,0.191586,Healthy


In [4]:
summary=pd.DataFrame({
    'Metric':[
        'Total Records',
        'Healthy Predictions',
        'Bankrupt Predictions',
        'Average Bankruptcy Probability'
    ],
    'Value':[
        len(results),
        (results['Prediction']==0).sum(),
        (results['Prediction']==1).sum(),
        round(results['Bankruptcy_Probability'].mean(),4)
    ]
})

summary.to_csv('batch_prediction_summary.csv',index=False)

summary


,Metric,Value
0,Total Records,78682.0000
1,Healthy Predictions,74203.0000
2,Bankrupt Predictions,4479.0000
3,Average Bankruptcy Probability,0.1147


## Deliverables

- `batch_prediction_results.csv`
- `batch_prediction_summary.csv`

These files can be delivered to business users, analysts, or downstream systems for decision-making and reporting.

## Executive Summary

This notebook demonstrates a production-style batch scoring pipeline. It loads the serialized model, scores every company in the cleaned dataset, generates bankruptcy probabilities, assigns prediction labels, and exports results for operational use.